# Inspect audio LSB sync

This notebook loads a WAV file recorded with Avisoft RECORDER USGH, extracts the least significant bit (LSB) from the audio samples, and plots the embedded digital sync track. Set `SESSION_BASEPATH` to the unzipped example session folder before running the notebook.

## Configure the input file

Use a 16-bit WAV file recorded with the UltraSoundGate digital input retained in the audio samples.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf


SESSION_BASEPATH = Path("/Users/xizheng/Documents/Mike/misc/260430_hardware_sync/test_sessions/example_session_01")
AUDIO_CHANNEL_DIR = "c1_high"
AUDIO_CHANNEL = 0

# Set to an integer to limit the number of points in full-session plots.
# Set to None to plot every sample.
FULL_PLOT_MAX_POINTS = 200_000

ZOOM_BEFORE_S = 0.15
ZOOM_AFTER_S = 0.35

## Load the audio and extract bit 0

`soundfile.read(..., dtype="int16")` preserves the integer sample values so the LSB can be extracted directly.

In [ ]:
audio_dir = SESSION_BASEPATH / "original_audio" / AUDIO_CHANNEL_DIR
audio_candidates = sorted(audio_dir.glob("*.wav"))
if not audio_candidates:
    raise FileNotFoundError(f"No WAV files found in: {audio_dir}")
AUDIO_WAV_PATH = audio_candidates[0]

if not AUDIO_WAV_PATH.exists():
    raise FileNotFoundError(f"Audio file not found: {AUDIO_WAV_PATH}")

audio, sample_rate = sf.read(AUDIO_WAV_PATH, dtype="int16", always_2d=True)
samples = audio[:, AUDIO_CHANNEL]

lsb = (samples & 1).astype(np.uint8)

time_s = np.arange(samples.size) / sample_rate
audio_float = samples.astype(np.float32) / np.iinfo(np.int16).max


def plot_indices(n_points, max_points=None):
    if max_points is None:
        return np.arange(n_points)
    stride = max(1, int(np.ceil(n_points / max_points)))
    return np.arange(0, n_points, stride)


print(f"file: {AUDIO_WAV_PATH}")
print(f"sample_rate: {sample_rate} Hz")
print(f"samples: {samples.size}")
print(f"duration: {samples.size / sample_rate:.3f} s")
print(f"channels: {audio.shape[1]}")
print(f"lsb states: {np.unique(lsb)}")

## Plot the full recording

The top panel shows the acoustic waveform. The bottom panel shows the extracted LSB sync track across the same time axis.

In [ ]:
full_idx = plot_indices(samples.size, FULL_PLOT_MAX_POINTS)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, constrained_layout=True)

axes[0].plot(time_s[full_idx], audio_float[full_idx], color="tab:blue")
axes[0].set_ylabel("audio")
axes[0].set_title("Audio waveform and embedded LSB sync")

axes[1].plot(time_s[full_idx], lsb[full_idx], color="tab:red")
axes[1].set_ylabel("LSB")
axes[1].set_xlabel("time (s)")
axes[1].set_yticks([0, 1])

plt.show()

## Zoom in around the sync onset

This plot zooms around the first detected LSB transition so the square-wave pulse train can be inspected directly.

In [ ]:
edges = np.flatnonzero(np.diff(lsb.astype(np.int8)) != 0) + 1
if edges.size == 0:
    raise ValueError("No LSB transitions detected. Check DIN wiring, RECORDER settings, and polarity.")

first_edge = edges[0]
zoom_start_s = max(0, time_s[first_edge] - ZOOM_BEFORE_S)
zoom_end_s = min(time_s[-1], time_s[first_edge] + ZOOM_AFTER_S)
zoom_mask = (time_s >= zoom_start_s) & (time_s <= zoom_end_s)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, constrained_layout=True)

axes[0].plot(time_s[zoom_mask], audio_float[zoom_mask], color='tab:blue')
axes[0].set_ylabel("audio")
axes[0].set_title("Zoom around first LSB transition")

axes[1].plot(time_s[zoom_mask], lsb[zoom_mask], color='tab:red')
axes[1].set_ylabel("LSB")
axes[1].set_xlabel("time (s)")
axes[1].set_yticks([0, 1])

plt.show()

## Full-session figure with inset zoom

This final plot is useful for the protocol figure: it shows the full audio recording and embedded LSB sync track, with an inset zoom around the first detected sync transition.

In [ ]:
full_idx = plot_indices(samples.size, FULL_PLOT_MAX_POINTS)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, constrained_layout=True)

axes[0].plot(time_s[full_idx], audio_float[full_idx], color='tab:blue')
axes[0].set_ylabel("audio")
axes[0].set_title("Audio waveform and embedded LSB sync")

axes[1].plot(time_s[full_idx], lsb[full_idx], color='tab:red')
axes[1].set_ylabel("LSB")
axes[1].set_xlabel("time (s)")
axes[1].set_yticks([0, 1])

inset_ax = axes[1].inset_axes([0.6, 0.35, 0.35, 0.5])
inset_ax.plot(time_s[zoom_mask], lsb[zoom_mask], color='tab:red')
inset_ax.set_xlim(zoom_start_s, zoom_end_s)
inset_ax.set_ylim(-0.1, 1.1)
inset_ax.set_xticks([])
inset_ax.set_yticks([0, 1])
inset_ax.set_title("sync onset")
axes[1].indicate_inset_zoom(inset_ax, edgecolor="gray", alpha=1)

plt.show()